# 08 · Explorar datos con pandas

Abre el notebook desde **PySpark/notebooks**. Los CSV están en **PySpark/data**. Ejecuta las celdas de arriba abajo. Necesitas pandas, matplotlib y SciPy en el kernel de Jupyter. Si faltan, puedes instalar las tres bibliotecas en una celda con %pip install pandas matplotlib scipy.

El **análisis exploratorio de datos (EDA)** empieza por entender qué representa cada fila y detectar qué debemos revisar. Después veremos tres preguntas de la encuesta sobre la forma de los salarios, la relación entre salario y kilómetros, y la asociación entre grupo profesional y tener coche.


## Los dos archivos

**[empleados_curso.csv](../data/empleados_curso.csv)** contiene 1.007 fichas de una encuesta de empleados adaptada para clase. Una fila es una ficha y una persona puede aparecer varias veces. Los nombres y DNI se retiraron.

| Columna | Qué significa |
|---|---|
| id_persona | Código de la persona encuestada. |
| id_registro | Código de la ficha. Las copias de una ficha comparten este código. |
| ciudad | Ciudad indicada en la encuesta. |
| grupo | Grupo profesional. Las categorías del ejercicio van de Grup A a Grup D. |
| salario_original | Salario mensual en euros tal como se registró. |
| tiene_coche | Respuesta sobre si tiene coche. True es sí. False es no. |
| km_anuales | Kilómetros recorridos en un año. |

**[ciudades_contexto.csv](../data/ciudades_contexto.csv)** contiene 36 filas. Cada fila asocia una ciudad con el país en el que está y con un índice de coste de vida de la tabla del curso.

| Columna | Qué significa |
|---|---|
| ciudad | Nombre de la ciudad en el archivo de contexto. |
| pais_contexto | País al que corresponde esa ciudad. |
| indice_coste_vida | Índice de la ciudad. Se expresa en puntos. |

Podemos usar la ciudad para juntar las tablas. Antes hay que decidir qué hacer con nombres como Paris, que aparece asociado a dos países.

## Guía para revisar cualquier dataset

| Paso | Qué mirar en cualquier dataset | Qué miraremos aquí |
|---|---|---|
| 1. Entender y abrir | Pregunta de análisis, fuente, qué representa una fila, columnas, unidades, tamaño y tipos. | Fichas de empleados y tabla de ciudades. |
| 2. Buscar ausencias y revisar categorías | Cuenta vacíos, revisa etiquetas y pregunta qué significa cada ausencia. | Coche sin respuesta, kilómetros vacíos y grupo Z. |
| 3. Revisar repeticiones | Decide qué identifica una fila y qué identifica a la persona u objeto que vas a contar. | Copias de fichas y personas con varias fichas. |
| 4. Revisar formatos y valores | Busca números guardados como texto, unidades mezcladas, grafías distintas y valores imposibles o extremos. | Salarios con símbolos, nombres de ciudades y kilómetros negativos. |
| 5. Combinar tablas si hace falta | Identifica la columna común, busca claves compartidas por varias filas y cuenta las coincidencias. | Paris y València/Valencia son ambiguas en el contexto. |
| 6. Resumir y comunicar | Mira recuentos, medidas típicas y relaciones. Anota qué cambiaste, cuántos datos usaste y qué límites tiene la conclusión. | Salario por grupo, respuestas sobre el coche y relación entre salario y kilómetros. |

Conserva la tabla original y realiza las correcciones en otra tabla. Las reglas de limpieza dependen de lo que significan los datos.

## 1. Abrir y mirar las dos tablas

Mostramos cinco filas para orientarnos. Después miramos el tamaño y los tipos de las columnas. El salario aparece como texto porque mezcla cifras con símbolos y palabras.

In [ ]:
import pandas as pd
from IPython.display import display

empleados = pd.read_csv('../data/empleados_curso.csv')
ciudades = pd.read_csv('../data/ciudades_contexto.csv')

display(empleados.head())
display(ciudades.head())
print('Empleados (filas, columnas):', empleados.shape)
print('Ciudades (filas, columnas):', ciudades.shape)
display(empleados.dtypes)
display(ciudades.dtypes)

## 2. Buscar datos vacíos y mirar categorías

Hay 349 fichas sin respuesta sobre el coche. Una respuesta vacía significa que desconocemos la respuesta. Un kilometraje vacío puede corresponder a alguien sin coche o a alguien que tenía coche y dejó el campo en blanco. También aparece un grupo Z, fuera de las categorías de la encuesta.

In [ ]:
display(empleados.isna().sum())
display(empleados['grupo'].value_counts(dropna=False))
display(empleados['tiene_coche'].value_counts(dropna=False))

## 3. Distinguir copias de fichas y personas repetidas

Quitamos las copias por el código de ficha. Apartamos las fichas sin código de persona. Si aún quedan varias fichas distintas para una persona, apartamos todas las suyas porque desconocemos cuál está vigente.

Compara los cuatro recuentos. La diferencia entre dos recuentos consecutivos te dice cuántas fichas apartaste en ese paso.

In [ ]:
sin_copias = empleados.drop_duplicates(subset='id_registro')
con_persona = sin_copias.dropna(subset=['id_persona'])
base = con_persona.drop_duplicates(subset='id_persona', keep=False).copy()

print('Al principio:', len(empleados))
print('Sin copias de ficha:', len(sin_copias))
print('Con código de persona:', len(con_persona))
print('Una ficha por persona:', len(base))

## 4. Limpiar formatos y valores

### Salario

En este CSV los importes escritos con cifras están en euros enteros. La coma y el punto separan los miles. Por ejemplo, «1,469 €» son 1469 euros. En otros archivos una coma o un punto puede marcar decimales.

Primero quitamos las marcas de moneda, los espacios y los separadores. Luego traducimos tres importes escritos en catalán. El resultado se guarda en una columna nueva. Las respuestas «n/d» y «molts» quedan sin importe.

In [ ]:
importes_escritos = {'mil vuit-cents': '1800', 'dos mil': '2000', 'tres mil': '3000'}

importe = base['salario_original'].str.lower()
importe = importe.str.replace(r'euros|â‚¬|[€,.]', '', regex=True).str.strip()
importe = importe.replace(importes_escritos)
base['salario_eur'] = pd.to_numeric(importe, errors='coerce')

display(base[['salario_original', 'salario_eur']].head(8))
display(base.loc[base['salario_eur'].isna(), 'salario_original'].value_counts(dropna=False))

### Grupo y kilometraje

Dejamos Z sin categoría. Un kilometraje negativo es imposible y queda vacío. También dejamos vacío el kilometraje cuando la persona responde que no tiene coche. Si la respuesta sobre el coche falta, conservamos el kilometraje registrado y dejamos esa ficha fuera de la comparación entre personas que declararon tener coche.

In [ ]:
base['grupo'] = base['grupo'].replace('Z', pd.NA)

sin_coche = base['tiene_coche'].eq(False)
km_negativos = base['km_anuales'] < 0

print('Kilometrajes negativos:', km_negativos.sum())
print('Kilometraje registrado pese a responder que no tiene coche:',
      (sin_coche & base['km_anuales'].notna()).sum())

base['km_limpios'] = base['km_anuales'].mask(sin_coche | km_negativos)

## 5. Añadir contexto de ciudad

En el archivo de contexto, A Coruña aparece escrita como «A Cor\u00f1a» y München como «Munchen». Igualamos esas grafías y corregimos caracteres dañados en la encuesta, como «ValÃ¨ncia». Para detectar el caso València/Valencia, usamos «Valencia» como nombre común en ambas tablas. Así quedan a la vista dos nombres ambiguos, Paris y Valencia. Apartamos sus entradas de la tabla de contexto porque la encuesta solo indica la ciudad.

La unión **izquierda** conserva las fichas de empleados. Donde falta una coincidencia, el índice queda vacío. Conservamos la grafía de la encuesta para leer los resultados.

In [ ]:
grafias = {r'A Cor\u00f1a': 'A Coruña', 'Munchen': 'München', 'València': 'Valencia'}

grafias_encuesta = {
    'A CoruÃ±a': 'A Coruña', 'MÃ¼nchen': 'München', 'ValÃ¨ncia': 'València',
    'MÃ\xa0laga': 'Málaga', 'DÃ¼sseldorf': 'Düsseldorf',
    'KÃ¶ln': 'Köln', 'TromsÃ¸': 'Tromsø',
}

contexto = ciudades.copy()
contexto['clave_ciudad'] = contexto['ciudad'].str.strip().replace(grafias)

ambiguas = contexto.duplicated(subset='clave_ciudad', keep=False)
display(contexto.loc[ambiguas, ['ciudad', 'pais_contexto']])

contexto_unico = contexto.drop_duplicates(subset='clave_ciudad', keep=False)
ciudad_limpia = base['ciudad'].str.strip().replace(grafias_encuesta)
ampliada = base.assign(clave_ciudad=ciudad_limpia.replace(grafias))
ampliada = ampliada.merge(contexto_unico[['clave_ciudad', 'indice_coste_vida']],
                          on='clave_ciudad', how='left')

display(ampliada[['ciudad', 'indice_coste_vida']].head(8))
print('Con índice:', ampliada['indice_coste_vida'].count())
print('Total de personas:', len(ampliada))

Stavanger y Málaga figuran en la encuesta y carecen de entrada en la tabla de contexto. Su índice queda vacío tras la unión. Compara el total de personas con el del paso 3 para ver qué ocurre con las filas.

## 6. Resumir y leer lo que encontramos

### ¿Cómo es el salario por grupo?

**Personas** cuenta todas las fichas del grupo. **Con salario** cuenta solo las que tienen un importe. Compara la media con la mediana, que es el valor central al ordenar los salarios. Mira el mínimo y el máximo para detectar valores que merezcan una segunda mirada.

In [ ]:
resumen = base.groupby('grupo')['salario_eur'].agg(
    personas='size', con_salario='count', media='mean', mediana='median',
    minimo='min', maximo='max', desviacion='std',
)
display(resumen.round(2))

### ¿Qué forma tienen los salarios? Normalidad con Shapiro–Wilk

Miramos los salarios del **grupo A** como ejemplo. El histograma agrupa los importes por intervalos y muestra cuántas personas hay en cada uno. Fíjate en si la distribución tiene una sola cima, es aproximadamente simétrica o tiene una cola más larga hacia un lado.

En Shapiro–Wilk, la **hipótesis nula (H₀)** dice que los salarios observados proceden de una población con distribución normal. En clase usaremos un umbral de **0,05**. Si el valor p es menor que 0,05, rechazamos H₀. Si es igual o mayor, la muestra aporta evidencia insuficiente para rechazarla. Lee el resultado junto con el histograma. Con muchas observaciones, el test puede detectar desviaciones pequeñas.

In [ ]:
from scipy import stats
import matplotlib.pyplot as plt

salarios_a = base.loc[base['grupo'] == 'Grup A', 'salario_eur'].dropna()

salarios_a.plot.hist(bins=20, title='Salarios del grupo A')
plt.xlabel('Salario mensual (€)')
plt.ylabel('Personas')
plt.show()

shapiro = stats.shapiro(salarios_a)
print('Salarios observados:', len(salarios_a))
print(f'Valor p de Shapiro–Wilk: {shapiro.pvalue:.3g}')

En esta encuesta, el histograma tiene una cola hacia los salarios altos. Con 446 salarios observados y un valor p de 3,64 × 10⁻²⁶, rechazamos la hipótesis de normalidad para el grupo A.

### ¿Se relacionan el salario y los kilómetros? Pearson y Spearman

Tomamos a quienes declararon tener coche y tienen los dos datos. En el gráfico, cada punto es una persona. Mira si los puntos suben, bajan o se dispersan y si hay casos alejados del resto.

- **Pearson** mide la relación lineal, la que se aproxima a una recta.
- **Spearman** compara las posiciones al ordenar cada medida y detecta tendencias que suben o bajan aunque la relación sea curva.

Los dos coeficientes van de −1 a +1. Aquí los usamos para describir los mismos pares de datos. Un valor cercano a cero indica poca relación del tipo que mide el coeficiente. Una asociación observada por sí sola no demuestra una causa.

In [ ]:
con_coche = base[base['tiene_coche'].eq(True)]
pares = con_coche[['salario_eur', 'km_limpios']].dropna()

pares.plot.scatter(x='salario_eur', y='km_limpios', alpha=0.5)
plt.xlabel('Salario mensual (€)')
plt.ylabel('Kilómetros anuales')
plt.show()

pearson = stats.pearsonr(pares['salario_eur'], pares['km_limpios'])
spearman = stats.spearmanr(pares['salario_eur'], pares['km_limpios'])

print('Personas que declararon tener coche:', len(con_coche))
print('Personas con ambos datos:', len(pares))
print(f'Pearson: {pearson[0]:.3f}')
print(f'Spearman: {spearman[0]:.3f}')

De 323 personas que declararon tener coche, 298 tienen los dos datos. Pearson (−0,096) y Spearman (−0,070) muestran una relación negativa débil en esos pares.

### ¿Se asocian el grupo y tener coche? Chi-cuadrado de independencia

Comparamos **Grup A** y **Grup B**. La tabla inicial cuenta los «sí», los «no» y las respuestas vacías. El porcentaje de «sí» se calcula sobre quienes respondieron.

La **hipótesis nula (H₀)** dice que el grupo profesional y la respuesta sobre el coche son independientes. Chi-cuadrado compara los recuentos observados con los que esperaríamos bajo H₀. Usamos las respuestas «sí» y «no» para esa comparación. Miramos cuántas respuestas faltan para entender a qué personas describe el resultado.

El contraste supone observaciones independientes. Como regla práctica, buscamos recuentos observados y esperados de al menos cinco por celda. Con el umbral de 0,05, un valor p menor aporta evidencia de asociación. Un valor igual o mayor indica evidencia insuficiente para rechazar la independencia.

In [ ]:
grupos_ab = base[base['grupo'].isin(['Grup A', 'Grup B'])]
respuesta = grupos_ab['tiene_coche'].map({True: 'si', False: 'no'}).fillna('sin_respuesta')

recuentos = pd.crosstab(grupos_ab['grupo'], respuesta)
recuentos['pct_si'] = 100 * recuentos['si'] / (recuentos['si'] + recuentos['no'])
display(recuentos.round(1))

observados = recuentos[['no', 'si']]
chi = stats.chi2_contingency(observados, correction=False)

esperados = pd.DataFrame(chi.expected_freq, index=observados.index, columns=observados.columns)
display(esperados.round(1))
print(f'Valor p de chi-cuadrado: {chi.pvalue:.3g}')

Declararon tener coche el 50,7 % de quienes respondieron en A y el 51,5 % en B. El valor p de 0,846 indica evidencia insuficiente para rechazar la independencia en esta comparación.

### Ampliación. ¿Cuánto difieren los salarios medios de A y B?

El contraste de **Welch** plantea que los dos grupos tienen la misma media poblacional y admite diferencias de dispersión. Aquí empezamos leyendo la diferencia observada en euros. El contraste supone observaciones independientes. La revisión de normalidad del grupo A describe un grupo y la interpretación de Welch depende de ambos grupos y del diseño de la encuesta.

In [ ]:
salarios_b = base.loc[base['grupo'] == 'Grup B', 'salario_eur'].dropna()
welch = stats.ttest_ind(salarios_a, salarios_b, equal_var=False)

print(f'Diferencia media A menos B: {salarios_a.mean() - salarios_b.mean():.2f} €')
print(f'Valor p de Welch: {welch.pvalue:.3g}')

**Para cerrar.** Explica qué representa una fila al empezar y al terminar. Describe dos decisiones de limpieza, cuántas personas entraron en cada análisis y qué puedes decir con esos datos. En un conjunto grande, los recuentos, filtros, uniones y resúmenes pueden hacerse con PySpark aplicando los mismos criterios.

[Guía de contrastes de Machine Learning Mastery](https://machinelearningmastery.com/statistical-hypothesis-tests-in-python-cheat-sheet/) · [Shapiro–Wilk](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.shapiro.html) · [Chi-cuadrado](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html)

## Navegación

[Anterior · Spark y HDFS](07_spark_hdfs.ipynb) · [Volver al README](../README.md) · [Siguiente · Visualización de datos](09_data_visualization.ipynb)